# adaptec1 - LLM-guided **LEGAL** macro placement vs MaskPlace (integrated greedy)

adaptec1 (ISPD-2005) has **543 macros + ~210k raw standard cells**. Here we place the **543 macros** with the same position-mask wiremask greedy used for ariane, so the layout is **legal by construction** (zero overlap, in-canvas). Standard cells are NOT placed.

We compare against **MaskPlace Table 2** (arXiv 2211.13382), which is also a **macro-only** MST wirelength at grid **N=224** -> no DREAMPlace needed (that is only for the paper's Table 6 full macro+std-cell number). adaptec1 is only ~48% dense, so all 543 fit legally at the native **grid 224**. We report 224 (headline) and a 448 sensitivity row.

Metric note: this uses the **identical** `place_db` + `comp_res` pipeline MaskPlace uses (693 macro-macro nets after std-cell pins are dropped), so MST is apples-to-apples.

## Cell 1 - Setup (clone repo, checkout branch, gym/protobuf/anthropic)

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
BRANCH    = "adaptec1-macro-compare"   # branch carrying the adaptec1 generalization
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"fetch","--all"]); print("fetched")
if subprocess.call(["git","-C",CLONE_DIR,"checkout",BRANCH]) == 0:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("on branch", BRANCH)
else:
    print("branch", BRANCH, "not found -- push it or merge to main")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    reg=types.ModuleType("gym.envs.registration"); envs=types.ModuleType("gym.envs")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    reg.register=lambda *a,**k:None; gym.envs=envs
    sys.modules.update({"gym":gym,"gym.spaces":spaces,"gym.envs":envs,"gym.envs.registration":reg})
    print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

need=["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
      "parse_netlist.py","region_constraint.py","strong_search.py","trade_off_eval.py",
      "integrated_search.py","adaptec1/adaptec1.nodes","adaptec1/adaptec1.nets",
      "adaptec1/adaptec1.pl"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push branch {BRANCH}): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 - Sanity (543 macros, macro-only netlist) + load PlaceDB

In [ ]:
from place_db import PlaceDB
placedb = PlaceDB("adaptec1")
areas = [placedb.node_info[n]["x"]*placedb.node_info[n]["y"] for n in placedb.node_info]
dens = 100*sum(areas)/(placedb.max_height*placedb.max_width)
print("Macros", len(placedb.node_info), "| macro-macro Nets", len(placedb.net_info),
      "| canvas", placedb.max_height, "x", placedb.max_width,
      "| continuous density %.1f%%" % dens)
assert len(placedb.node_info) == 543
print("sanity OK  (543 macros; ~210k std cells are NOT placed -- macro-only, MaskPlace Table 2)")

## Cell 3 - API key (needed only for the LLM cell)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

## Cell 4 - Config

In [ ]:
import importlib, integrated_search
importlib.reload(integrated_search)
import integrated_search as isr

BENCHMARK  = "adaptec1"         # selects the MaskPlace Table-2 reference (isr.MASKPLACE_REF)
TOP_N      = 543                # all 543 macros are reorder-able by the LLM/random search
GRID       = 224               # MaskPlace native grid; all 543 fit LEGALLY (~48% dense)
GRID_SENS  = 448               # sensitivity row (baseline only)
MODEL      = "claude-opus-4-8"  # strongest model; tiny output (promote moves), cached prompt
MAX_ITERS  = 8
PATIENCE   = 3
N_RANDOM   = 12
N_MOVES    = 8                  # max targeted moves/step (LLM and matched random control)
LLM_ACTION = "promote"          # option C: targeted 'place A before B' moves on topology order
print(f"benchmark={BENCHMARK} | action={LLM_ACTION} | reorder-able: top {TOP_N} of 543 | "
      f"grid {GRID} (+{GRID_SENS} sens) | n_moves {N_MOVES} | model {MODEL}")
print(f"MaskPlace ref (Table 2, MST, grid 224): {isr.MASKPLACE_REF[BENCHMARK]:.3e}")

## Cell 5 - Heuristic baseline + random control (FREE, no API) -- LEGAL 543

In [ ]:
# FREE (no API): topology baseline (grid 224 + 448 sensitivity) + matched random promote control.
# All place the 543 macros with the position-mask greedy -> LEGAL (zero overlap) by construction.
base    = isr.heuristic_baseline(placedb, grid=GRID, verbose=True)
base448 = isr.heuristic_baseline(placedb, grid=GRID_SENS, verbose=True)   # sensitivity
rc = isr.random_promote_control(placedb, top_n=TOP_N, grid=GRID, n_evals=N_RANDOM,
                                n_moves=N_MOVES, verbose=True)

## Cell 6 - LLM promote search (PAID, text) -- option C -- LEGAL 543

In [ ]:
# PAID (text): the LLM proposes up to N_MOVES targeted moves on the topology order; every
# candidate scored on the integrated LEGAL 543 layout (accept-if-improves, floored at topology).
res = isr.llm_promote_search(placedb, top_n=TOP_N, grid=GRID, model=MODEL,
                             max_iters=MAX_ITERS, patience=PATIENCE, max_moves=N_MOVES,
                             verbose=True)

## Cell 7 - Results table (legality + MaskPlace comparison) + CSV

In [ ]:
# Combined legality + comparison table (run after whichever cells above you ran).
# Reports BOTH bbox HPWL and MST; compare the MST column to MaskPlace (its number is MST).
import csv
from trade_off_eval import usd_cost
N_TOTAL = len(placedb.node_info)

def _row(method, r, grid, calls=0, intok=0, outok=0, ctok=0):
    ov, oob = isr.overlaps_and_oob(r["best_env"], grid)
    return dict(method=method, hpwl=r["hpwl"], mst=isr.mst_of(placedb, r["best_env"]),
                overlaps=ov, out_of_canvas=oob, macros=len(r["best_env"].node_pos), grid=grid,
                calls=calls, out_tokens=outok,
                usd=(usd_cost(MODEL, intok, outok, ctok) if calls else 0.0))

rows = []
if "base"    in globals(): rows.append(_row("heuristic baseline", base, GRID))
if "rc"      in globals(): rows.append(_row("random promote ctrl", rc, GRID))
if "res"     in globals(): rows.append(_row("LLM promote", res, GRID, calls=res["calls"],
                                            intok=res["in_tokens"], outok=res["out_tokens"],
                                            ctok=res["cache_read_tokens"]))
if "base448" in globals(): rows.append(_row(f"baseline (grid {GRID_SENS})", base448, GRID_SENS))
rows.append(dict(method="MaskPlace RL (paper)", hpwl=None, mst=isr.MASKPLACE_REF[BENCHMARK],
                 overlaps=0, out_of_canvas=0, macros=N_TOTAL, grid=224, calls=0, out_tokens=0,
                 usd=None))

def _f(v): return f"{v:.3e}" if v is not None else "-"
print(f"\n{'method':<22}{'HPWL(bbox)':>12}{'MST':>12}{'macros':>8}{'overlaps':>9}{'oob':>5}{'grid':>6}{'$':>8}{'out_tok':>9}")
print("-"*92)
for r in rows:
    usd = f"{r['usd']:.3f}" if r["usd"] is not None else "-"
    inc = r["macros"] < N_TOTAL and r["method"] != "MaskPlace RL (paper)"
    hb = "incompl" if inc else _f(r["hpwl"])
    mb = "incompl" if inc else _f(r["mst"])
    print(f"{r['method']:<22}{hb:>12}{mb:>12}{r['macros']:>8}{r['overlaps']:>9}"
          f"{r['out_of_canvas']:>5}{r['grid']:>6}{usd:>8}{r['out_tokens']:>9}")
print(f"\nMST column is metric-matched to MaskPlace (its {isr.MASKPLACE_REF[BENCHMARK]:.3e} is MST, grid 224,")
print("macro-only). Our rows are legal-by-construction (0 overlap). Judge the LLM vs the random control.")

with open("/kaggle/working/adaptec1_macro.csv","w",newline="") as f:
    w=csv.DictWriter(f, fieldnames=["method","hpwl","mst","overlaps","out_of_canvas","macros","grid","calls","out_tokens","usd"])
    w.writeheader(); w.writerows(rows)
print("saved /kaggle/working/adaptec1_macro.csv")

## Cell 8 - Visualize the legal placement (543 macros)

In [ ]:
# Visualize a legal placement (run after a search cell). adaptec1 has no hard/soft tag, so all
# 543 macros are drawn in one color. Uses the LLM result if present, else random, else baseline.
src = res if "res" in globals() else (rc if "rc" in globals() else base)
isr.plot_placement(placedb, src["best_env"], grid=GRID,
                   title=f"adaptec1 legal placement ({len(src['best_env'].node_pos)} macros @ grid {GRID})",
                   path="/kaggle/working/adaptec1_placement.png")